# 연도별 Spearman 상관분석

## 분석 목적

이 노트북은 seed dictionary와 threshold별 expanded dictionary 점수가 KCGS ESG 통합등급과 갖는 Spearman 순위상관이 연도별로 일관적인지 확인한다.

## 연도 정렬 기준

DART 사업보고서 텍스트는 `fiscal_year` 기준이고, ESG 등급은 다음 해 평가값을 사용한다. 따라서 분석에서는 다음 관계를 명시적으로 적용한다.

```text
esg_year = fiscal_year + 1
```

예를 들어 2022년 사업보고서 텍스트는 2023년 ESG 등급과 매칭한다.

## 해석 기준

전체 표본 분석에서 seed dictionary의 `count`가 가장 높은 절대 Spearman 상관을 보였으므로, 연도별 분석은 이 결론이 특정 연도에만 의존하는지 확인하는 강건성 분석으로 사용한다.

In [1]:
import subprocess
import sys
from pathlib import Path
import re
import html
import unicodedata
import warnings

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "pandas",
    "numpy",
    "scikit-learn",
    "scipy",
    "statsmodels",
])

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 120)

print("packages loaded")


packages loaded


In [2]:
# Path setup: supports both Colab and local execution.
if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")

LOCAL_ROOT = Path.cwd()
DRIVE_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/My Drive/UD_26"),
]
ROOT_CANDIDATES = DRIVE_ROOT_CANDIDATES + [LOCAL_ROOT, LOCAL_ROOT.parent]


def first_existing(candidates, default=None):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return default if default is not None else candidates[0]


ROOT = first_existing(
    [p for p in ROOT_CANDIDATES if (p / "data").exists() or (p / "final").exists()],
    LOCAL_ROOT,
)

FINAL_DIR = ROOT / "final"
DATA_DIR = ROOT / "data"
DART_DIR = DATA_DIR / "dart"

COMPANY_MASTER_PATH = first_existing([
    DATA_DIR / "company_master.csv",
    FINAL_DIR / "company_master.csv",
])

FILING_INDEX_PATH = first_existing([
    DART_DIR / "filing_index.csv",
    FINAL_DIR / "filing_index.csv",
    DATA_DIR / "filing_index.csv",
])

RAW_XML_DIR = first_existing([
    DART_DIR / "raw_xml",
    FINAL_DIR / "raw_xml",
    ROOT / "raw_xml",
])

SEED_DICTIONARY_PATH = first_existing([
    FINAL_DIR / "seed_dictionary.csv",
    DATA_DIR / "seed_dictionary.csv",
])

EXPANDED_OUTPUT_DIR = FINAL_DIR / "expanded_dictionaries"
EXPANDED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for label, path in {
    "ROOT": ROOT,
    "COMPANY_MASTER_PATH": COMPANY_MASTER_PATH,
    "FILING_INDEX_PATH": FILING_INDEX_PATH,
    "RAW_XML_DIR": RAW_XML_DIR,
    "SEED_DICTIONARY_PATH": SEED_DICTIONARY_PATH,
    "EXPANDED_OUTPUT_DIR": EXPANDED_OUTPUT_DIR,
}.items():
    print(f"{label}: {path} | {'OK' if path.exists() or label == 'EXPANDED_OUTPUT_DIR' else 'MISSING'}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT: /content/drive/MyDrive/UD_26 | OK
COMPANY_MASTER_PATH: /content/drive/MyDrive/UD_26/data/company_master.csv | OK
FILING_INDEX_PATH: /content/drive/MyDrive/UD_26/final/filing_index.csv | OK
RAW_XML_DIR: /content/drive/MyDrive/UD_26/final/raw_xml | OK
SEED_DICTIONARY_PATH: /content/drive/MyDrive/UD_26/final/seed_dictionary.csv | OK
EXPANDED_OUTPUT_DIR: /content/drive/MyDrive/UD_26/final/expanded_dictionaries | OK


In [3]:
# Leave as None to score every expanded_dictionary_theta_*.csv file found in EXPANDED_OUTPUT_DIR.
# Set to a list to restrict scoring to selected thresholds.
REQUESTED_THRESHOLDS = [round(theta, 2) for theta in np.arange(0.55, 0.75 + 0.001, 0.01)]

GRADE_MAP = {"D": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5, "S": 6}

print("REQUESTED_THRESHOLDS:", REQUESTED_THRESHOLDS if REQUESTED_THRESHOLDS is not None else "all available")

REQUESTED_THRESHOLDS: [np.float64(0.55), np.float64(0.56), np.float64(0.57), np.float64(0.58), np.float64(0.59), np.float64(0.6), np.float64(0.61), np.float64(0.62), np.float64(0.63), np.float64(0.64), np.float64(0.65), np.float64(0.66), np.float64(0.67), np.float64(0.68), np.float64(0.69), np.float64(0.7), np.float64(0.71), np.float64(0.72), np.float64(0.73), np.float64(0.74), np.float64(0.75)]


In [4]:
company_master = pd.read_csv(COMPANY_MASTER_PATH, dtype={"stock_code": "string"}, encoding="utf-8-sig")
filing_index = pd.read_csv(FILING_INDEX_PATH, dtype={"stock_code": "string", "rcept_no": "string"}, encoding="utf-8-sig")
seed_df = pd.read_csv(SEED_DICTIONARY_PATH, encoding="utf-8-sig")

for df in [company_master, filing_index]:
    df["stock_code"] = df["stock_code"].astype("string").str.zfill(6)

seed_df = seed_df.copy()
seed_df["dimension"] = seed_df["dimension"].astype(str).str.strip()
seed_df["seed_term"] = seed_df["seed_term"].astype(str).str.strip()
seed_df = seed_df[seed_df["dimension"].isin(["E", "S", "G"])].drop_duplicates(["dimension", "seed_term"])

print("company_master:", company_master.shape)
print("filing_index:", filing_index.shape)
print("seed_df:", seed_df.shape)
display(seed_df[["dimension", "seed_term", "pattern"]].head(20))
display(seed_df["dimension"].value_counts().rename_axis("dimension").reset_index(name="seed_count"))


company_master: (381, 13)
filing_index: (381, 13)
seed_df: (30, 6)


,dimension,seed_term,pattern
0,E,탄소,탄소
1,E,온실가스,온실가스|GHG
2,E,탄소중립,탄소중립
3,E,넷제로,넷제로|net zero|net-zero
4,E,재생에너지,재생에너지|renewable energy
5,E,에너지,에너지
6,E,전력,전력|전력사용량
7,E,폐기물,폐기물
8,E,재활용,재활용|자원순환
9,E,폐수,폐수|수질|물관리


,dimension,seed_count
0,E,10
1,S,10
2,G,10


In [5]:
def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKC", html.unescape(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_term(term):
    return normalize_text(term)


def threshold_label(theta):
    return f"{theta:.2f}".replace(".", "_")


def dictionary_name(theta):
    return f"expanded_dictionary_theta_{threshold_label(theta)}"


def pattern_from_term(term):
    return re.escape(normalize_term(term))


def threshold_from_dictionary_path(path):
    match = re.fullmatch(r"expanded_dictionary_theta_(\d+)_(\d+)", path.stem)
    if match is None:
        raise ValueError(f"Cannot parse threshold from expanded dictionary filename: {path.name}")
    return float(f"{int(match.group(1))}.{match.group(2)}")

print("helpers ready")


helpers ready


In [6]:
def build_seed_dictionary_frame(seed_df):
    rows = []
    for _, row in seed_df.iterrows():
        values = [row.get("seed_term", "")]
        pattern = row.get("pattern", "")
        if pd.notna(pattern):
            values.extend(str(pattern).split("|"))
        seen = set()
        for value in values:
            term = normalize_term(value)
            if not term or term.lower() == "nan" or term in seen:
                continue
            seen.add(term)
            rows.append({
                "dictionary_name": "seed_dictionary",
                "threshold": np.nan,
                "dimension": row["dimension"],
                "seed_term": normalize_term(row["seed_term"]),
                "candidate_term": term,
                "pattern": pattern_from_term(term),
                "similarity": 1.0,
                "source": "seed",
                "seed_terms_matched": normalize_term(row["seed_term"]),
                "keep_review": True,
                "exclude_reason": "",
            })
    return pd.DataFrame(rows).drop_duplicates(["dimension", "candidate_term"]).reset_index(drop=True)


def load_seed_dictionary(seed_df):
    seed_dictionary_path = EXPANDED_OUTPUT_DIR / "seed_dictionary_normalized_for_comparison.csv"
    if seed_dictionary_path.exists():
        seed_dictionary = pd.read_csv(seed_dictionary_path, encoding="utf-8-sig")
        print("seed dictionary loaded:", seed_dictionary_path)
    else:
        seed_dictionary = build_seed_dictionary_frame(seed_df)
        seed_dictionary.to_csv(seed_dictionary_path, index=False, encoding="utf-8-sig")
        print("seed dictionary built from seed_df and saved:", seed_dictionary_path)
    return seed_dictionary


def discover_expanded_dictionary_files(requested_thresholds=None):
    files_by_threshold = {
        threshold_from_dictionary_path(path): path
        for path in sorted(EXPANDED_OUTPUT_DIR.glob("expanded_dictionary_theta_*.csv"))
    }
    if not files_by_threshold:
        raise FileNotFoundError(
            "No expanded dictionary CSVs found in "
            f"{EXPANDED_OUTPUT_DIR}. Run the new generation notebook first to create "
            "expanded_dictionary_theta_*.csv files."
        )

    if requested_thresholds is None:
        return files_by_threshold

    requested = [float(theta) for theta in requested_thresholds]
    missing = [theta for theta in requested if theta not in files_by_threshold]
    if missing:
        available = ", ".join(f"{theta:.2f}" for theta in sorted(files_by_threshold))
        missing_text = ", ".join(f"{theta:.2f}" for theta in missing)
        raise FileNotFoundError(
            f"Requested expanded dictionary thresholds not found: {missing_text}. "
            f"Available thresholds in {EXPANDED_OUTPUT_DIR}: {available}."
        )
    return {theta: files_by_threshold[theta] for theta in requested}


def load_expanded_dictionaries(requested_thresholds=None):
    files_by_threshold = discover_expanded_dictionary_files(requested_thresholds)
    dictionary_map = {}
    loaded_rows = []
    for theta, path in sorted(files_by_threshold.items()):
        df = pd.read_csv(path, encoding="utf-8-sig")
        name = dictionary_name(theta)
        dictionary_map[name] = df
        loaded_rows.append({
            "threshold": theta,
            "dictionary_name": name,
            "path": path,
            "rows": len(df),
        })
    return dictionary_map, pd.DataFrame(loaded_rows)


seed_dictionary_df = load_seed_dictionary(seed_df)
expanded_dictionary_map, expanded_dictionary_files_df = load_expanded_dictionaries(REQUESTED_THRESHOLDS)
THRESHOLDS = expanded_dictionary_files_df["threshold"].tolist()

print("discovered THRESHOLDS:", THRESHOLDS)
display(expanded_dictionary_files_df)


seed dictionary loaded: /content/drive/MyDrive/UD_26/final/expanded_dictionaries/seed_dictionary_normalized_for_comparison.csv
discovered THRESHOLDS: [0.55, 0.56, 0.57, 0.58, 0.59, 0.6, 0.61, 0.62, 0.63, 0.64, 0.65, 0.66, 0.67, 0.68, 0.69, 0.7, 0.71, 0.72, 0.73, 0.74, 0.75]


,threshold,dictionary_name,path,rows
0,0.55,expanded_dictionary_theta_0_55,/content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_55.csv,2469
1,0.56,expanded_dictionary_theta_0_56,/content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_56.csv,2114
2,0.57,expanded_dictionary_theta_0_57,/content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_57.csv,1793
3,0.58,expanded_dictionary_theta_0_58,/content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_58.csv,1524
4,0.59,expanded_dictionary_theta_0_59,/content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_59.csv,1315
5,0.60,expanded_dictionary_theta_0_60,/content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_60.csv,1158
6,0.61,expanded_dictionary_theta_0_61,/content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_61.csv,1002
7,0.62,expanded_dictionary_theta_0_62,/content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_62.csv,835
8,0.63,expanded_dictionary_theta_0_63,/content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_63.csv,669
9,0.64,expanded_dictionary_theta_0_64,/content/drive/MyDrive/UD_26/final/expanded_dictionaries/expanded_dictionary_theta_0_64.csv,530


In [7]:
def strip_tags_keep_space(xml_fragment):
    text = html.unescape(str(xml_fragment))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return unicodedata.normalize("NFKC", text)


title_re = re.compile(r"<TITLE\b[^>]*>(.*?)</TITLE>", flags=re.I | re.S)
main_title_re = re.compile(r"^\s*(I|II|III|IV|V|VI|VII|VIII|IX|X|Ⅰ|Ⅱ|Ⅲ|Ⅳ|Ⅴ|Ⅵ|Ⅶ|Ⅷ|Ⅸ|Ⅹ)\.")
target_title_regex = {
    "II. 사업의 내용": r"^(II|Ⅱ)\.\s*사업의\s*내용",
    "IV. 이사의 경영진단 및 분석의견": r"^(IV|Ⅳ)\.\s*이사의\s*경영진단\s*및\s*분석의견",
    "VI. 이사회 등 회사의 기관에 관한 사항": r"^(VI|Ⅵ)\.\s*이사회\s*등\s*회사의\s*기관에\s*관한\s*사항",
}


def extract_target_sections(xml_text):
    matches = list(title_re.finditer(xml_text))
    sections = []
    active = None

    for i, match in enumerate(matches):
        title = strip_tags_keep_space(match.group(1))
        next_start = matches[i + 1].start() if i + 1 < len(matches) else len(xml_text)
        body = xml_text[match.end():next_start]

        if main_title_re.search(title):
            active = None
            for section_name, pattern in target_title_regex.items():
                if re.search(pattern, title):
                    active = section_name
                    break
            if active is None:
                continue

        if active:
            section_text = strip_tags_keep_space(body)
            if section_text:
                sections.append({"section": active, "title": title, "text": section_text})

    return sections

section_rows = []
missing_xml_rows = []

for _, row in filing_index.iterrows():
    xml_path_value = row.get("xml_path", "")
    xml_path_text = str(xml_path_value).replace("\\", "/")
    xml_path_candidates = [
        ROOT / xml_path_text,
        Path(xml_path_text),
        RAW_XML_DIR / Path(xml_path_text).name,
        FINAL_DIR / "raw_xml" / Path(xml_path_text).name,
        DART_DIR / "raw_xml" / Path(xml_path_text).name,
    ]
    xml_path = first_existing(xml_path_candidates, xml_path_candidates[0])
    if not xml_path.exists():
        missing_xml_rows.append({**row.to_dict(), "resolved_xml_path": str(xml_path)})
        continue

    xml_text = xml_path.read_text(encoding="utf-8", errors="ignore")
    for section in extract_target_sections(xml_text):
        section_rows.append({
            "stock_code": row["stock_code"],
            "company_name": row.get("company_name", ""),
            "fiscal_year": int(row["fiscal_year"]),
            "esg_year": int(row["esg_year"]),
            "rcept_no": row["rcept_no"],
            "section": section["section"],
            "section_text": normalize_text(section["text"]),
        })

section_df = pd.DataFrame(section_rows)
missing_xml_df = pd.DataFrame(missing_xml_rows)

print("section rows:", section_df.shape)
print("missing xml rows:", missing_xml_df.shape)
if not section_df.empty:
    display(section_df.groupby("section").size().rename("rows"))
    display(section_df.head())
else:
    raise ValueError("No target sections extracted. Check filing_index xml_path and section extraction rules.")


section rows: (4155, 7)
missing xml rows: (3, 14)


,rows
section,
II. 사업의 내용,2643
IV. 이사의 경영진단 및 분석의견,378
VI. 이사회 등 회사의 기관에 관한 사항,1134


,stock_code,company_name,fiscal_year,esg_year,rcept_no,section,section_text
0,005930,삼성전자,2022,2022,20230307000542,II. 사업의 내용,"당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개 지역총괄의 생산ᆞ판매법인, SDC 및 Harman 산하 종속기업 등 232개의 종속기업으로 구성된 글로벌 전자 기업입니다. 사업별로 보면, Set 사업은 TV를 비롯하여 모니터, 냉장..."
1,005930,삼성전자,2022,2022,20230307000542,II. 사업의 내용,"가. 주요 제품 매출 당사는 TV, 냉장고, 세탁기, 에어컨, HHP 등 완제품과 DRAM, NAND Flash, 모바일AP 등 반도체 부품 및 디스플레이 패널 등을 생산ᆞ판매하고 있습니다. 아울러 Harman을 통해 디지털 콕핏, 텔레매틱스 등을 생산ᆞ판매하고 있습니다.2022..."
2,005930,삼성전자,2022,2022,20230307000542,II. 사업의 내용,"가. 주요 원재료 현황 당사의 주요 원재료로 DX 부문은 모바일AP, Camera Module 등을 Qualcomm, 삼성전기(주) 등에서 공급받고 있으며, TVᆞ모니터용 디스플레이 패널 등을 CSOT 등으로부터 공급받고 있습니다. DS 부문은 Chemical, Wafer 등을 ..."
3,005930,삼성전자,2022,2022,20230307000542,II. 사업의 내용,"가. 매출실적 2022년 매출은 302조 2,314억원으로 전년 대비 8.1% 증가하였습니다. 부문별로 보면 전년 대비 DX 부문이 9.8%, DS 부문이 3.2% 증가하였으며, SDC가 8.4%, Harman이 31.6% 증가하였습니다. (단위 : 억원) 부 문 매출유형 품 목..."
4,005930,삼성전자,2022,2022,20230307000542,II. 사업의 내용,"가. 재무위험관리정책 당사의 재무위험관리는 영업 활동에서 파생되는 시장위험, 신용위험, 유동성위험 등을 최소화하는데 중점을 두고 있습니다. 당사는 이를 위해 각각의 위험요인에 대해 면밀하게 모니터링하고 대응하는 재무위험관리정책과 프로그램을 운영하고 있습니다. 위험을 회피하기 위한..."


In [8]:
corpus_df = (
    section_df.groupby(["stock_code", "company_name", "fiscal_year", "esg_year", "rcept_no"], as_index=False)
    .agg(
        document=("section_text", " ".join),
        section_count=("section", "nunique"),
    )
)
corpus_df["document_norm"] = corpus_df["document"].map(normalize_text)
corpus_df["total_char_count"] = corpus_df["document_norm"].str.len()
corpus_df["total_word_count"] = corpus_df["document_norm"].str.split().str.len()

print("firm-year corpus rows:", len(corpus_df))
display(corpus_df[["stock_code", "company_name", "fiscal_year", "esg_year", "section_count", "total_word_count"]].head())
display(corpus_df[["section_count", "total_word_count", "total_char_count"]].describe().T)


firm-year corpus rows: 378


,stock_code,company_name,fiscal_year,esg_year,section_count,total_word_count
0,000020,동화약품,2022,2022,3,7808
1,000020,동화약품,2023,2023,3,8065
2,000020,동화약품,2024,2024,3,8097
3,000040,KR모터스,2022,2022,3,4201
4,000040,KR모터스,2023,2023,3,4888


,count,mean,std,min,25%,50%,75%,max
section_count,378.0,3.000000,0.000000,3.0,3.0,3.0,3.00,3.0
total_word_count,378.0,14965.915344,12333.345076,1774.0,7611.5,11232.5,17153.75,71704.0
total_char_count,378.0,72960.116402,61337.067441,8657.0,36647.0,55039.5,83135.25,360632.0


In [9]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer


def prepare_dictionary_records(dictionary_df, label):
    usable = dictionary_df.copy()
    if "keep_review" in usable.columns:
        usable = usable[usable["keep_review"].astype(str).str.upper() != "FALSE"]
    usable = usable[usable["dimension"].isin(["E", "S", "G"])]
    usable["candidate_term"] = usable["candidate_term"].map(normalize_term)
    usable = usable[usable["candidate_term"].str.len() > 0]
    usable = usable.drop_duplicates(["dimension", "candidate_term"]).reset_index(drop=True)

    records = []
    for idx, row in usable.iterrows():
        term_id = f"{label}_{row['dimension']}_{idx:05d}"
        pattern = row.get("pattern", "")
        if pd.isna(pattern) or not str(pattern).strip():
            pattern = pattern_from_term(row["candidate_term"])
        records.append({
            "term_id": term_id,
            "dimension": row["dimension"],
            "candidate_term": row["candidate_term"],
            "source": row.get("source", ""),
            "similarity": row.get("similarity", np.nan),
            "regex": re.compile(str(pattern), flags=re.I),
        })
    return records


def score_dictionary(corpus_df, dictionary_df, label):
    records = prepare_dictionary_records(dictionary_df, label)
    if not records:
        raise ValueError(f"No usable dictionary terms for {label}")

    def dictionary_analyzer(text):
        text = normalize_text(text)
        tokens = []
        for rec in records:
            hit_count = len(rec["regex"].findall(text))
            if hit_count:
                tokens.extend([rec["term_id"]] * hit_count)
        return tokens

    count_vectorizer = CountVectorizer(analyzer=dictionary_analyzer, lowercase=False)
    X_count = count_vectorizer.fit_transform(corpus_df["document_norm"].fillna(""))
    feature_names = count_vectorizer.get_feature_names_out()

    presence_vectorizer = CountVectorizer(
        analyzer=dictionary_analyzer,
        lowercase=False,
        binary=True,
        vocabulary=count_vectorizer.vocabulary_,
    )
    X_presence = presence_vectorizer.transform(corpus_df["document_norm"].fillna(""))
    X_count_tfidf = TfidfTransformer(norm=None, use_idf=True, smooth_idf=True).fit_transform(X_count)
    X_presence_tfidf = TfidfTransformer(norm=None, use_idf=True, smooth_idf=True).fit_transform(X_presence)

    feature_df = pd.DataFrame({"term_id": feature_names})
    record_df = pd.DataFrame([{k: v for k, v in rec.items() if k != "regex"} for rec in records])
    feature_df = feature_df.merge(record_df, on="term_id", how="left")
    term_id_to_idx = {term_id: idx for idx, term_id in enumerate(feature_names)}

    out = corpus_df[["stock_code", "company_name", "fiscal_year", "esg_year", "rcept_no", "section_count", "total_word_count", "total_char_count"]].copy()
    word_denominator = out["total_word_count"].where(out["total_word_count"] != 0)

    dimensions = ["E", "S", "G"]
    for dimension in dimensions:
        term_ids = feature_df.loc[feature_df["dimension"] == dimension, "term_id"].tolist()
        cols = [term_id_to_idx[term_id] for term_id in term_ids if term_id in term_id_to_idx]
        if cols:
            out[f"{dimension}_{label}_count"] = X_count[:, cols].sum(axis=1).A1
            out[f"{dimension}_{label}_presence_count"] = X_presence[:, cols].sum(axis=1).A1
            out[f"{dimension}_{label}_count_tfidf_score"] = X_count_tfidf[:, cols].sum(axis=1).A1
            out[f"{dimension}_{label}_presence_tfidf_score"] = X_presence_tfidf[:, cols].sum(axis=1).A1
        else:
            out[f"{dimension}_{label}_count"] = 0.0
            out[f"{dimension}_{label}_presence_count"] = 0.0
            out[f"{dimension}_{label}_count_tfidf_score"] = 0.0
            out[f"{dimension}_{label}_presence_tfidf_score"] = 0.0
        out[f"{dimension}_{label}_share"] = out[f"{dimension}_{label}_count"] / word_denominator

    for metric in ["count", "presence_count", "count_tfidf_score", "presence_tfidf_score", "share"]:
        out[f"ESG_{label}_{metric}"] = sum(out[f"{dimension}_{label}_{metric}"] for dimension in dimensions)

    term_summary = feature_df.groupby(["dimension", "source"], dropna=False).size().rename("terms_observed").reset_index()
    hit_summary = []
    for dimension in dimensions:
        hit_summary.append({
            "dimension": dimension,
            "dictionary_label": label,
            "dictionary_terms": int(sum(1 for rec in records if rec["dimension"] == dimension)),
            "observed_terms": int((feature_df["dimension"] == dimension).sum()),
            "documents_with_any_hit": int((out[f"{dimension}_{label}_count"] > 0).sum()),
        })
    hit_summary = pd.DataFrame(hit_summary)

    return out, feature_df, term_summary, hit_summary

print("scoring function ready")


scoring function ready


In [10]:
from joblib import Parallel, delayed
import os


def score_one_dictionary(theta):
    label = f"expanded_{threshold_label(theta)}"
    name = dictionary_name(theta)
    score_df, feature_df, term_summary, hit_summary = score_dictionary(
        corpus_df,
        expanded_dictionary_map[name],
        label,
    )
    return theta, label, score_df, feature_df, term_summary, hit_summary


score_frames = []
term_summaries = []
hit_summaries = []
feature_maps = {}

seed_score_df, seed_feature_df, seed_term_summary, seed_hit_summary = score_dictionary(corpus_df, seed_dictionary_df, "seed")
score_frames.append(seed_score_df)
term_summaries.append(seed_term_summary.assign(dictionary_label="seed"))
hit_summaries.append(seed_hit_summary)
feature_maps["seed"] = seed_feature_df

# Colab notebooks are usually most stable with thread-based parallelism.
# Increase n_jobs cautiously if the runtime has enough CPU and memory.
n_jobs = min(4, os.cpu_count() or 1)
print(f"Scoring {len(THRESHOLDS)} expanded dictionaries with n_jobs={n_jobs}")

results = Parallel(n_jobs=n_jobs, prefer="threads", verbose=10)(
    delayed(score_one_dictionary)(theta)
    for theta in THRESHOLDS
)

for theta, label, score_df, feature_df, term_summary, hit_summary in results:
    score_frames.append(score_df)
    term_summaries.append(term_summary.assign(dictionary_label=label, threshold=theta))
    hit_summaries.append(hit_summary.assign(threshold=theta))
    feature_maps[label] = feature_df

base_cols = ["stock_code", "company_name", "fiscal_year", "esg_year", "rcept_no", "section_count", "total_word_count", "total_char_count"]
all_score_df = score_frames[0]
for frame in score_frames[1:]:
    metric_cols = [col for col in frame.columns if col not in base_cols]
    all_score_df = all_score_df.merge(frame[base_cols[:5] + metric_cols], on=base_cols[:5], how="left")

term_summary_df = pd.concat(term_summaries, ignore_index=True)
hit_summary_df = pd.concat(hit_summaries, ignore_index=True)

print("all_score_df:", all_score_df.shape)
display(hit_summary_df)
display(all_score_df.head())

Scoring 21 expanded dictionaries with n_jobs=2


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed: 10.7min
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed: 20.2min
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed: 32.9min
[Parallel(n_jobs=2)]: Done  14 tasks      | elapsed: 35.8min


all_score_df: (378, 448)


[Parallel(n_jobs=2)]: Done  21 out of  21 | elapsed: 37.8min finished


,dimension,dictionary_label,dictionary_terms,observed_terms,documents_with_any_hit,threshold
0,E,seed,18,18,304,NaN
1,S,seed,18,18,378,NaN
2,G,seed,17,15,378,NaN
3,E,expanded_0_55,954,152,329,0.55
4,S,expanded_0_55,830,148,378,0.55
5,G,expanded_0_55,685,93,378,0.55
6,E,expanded_0_56,916,128,322,0.56
7,S,expanded_0_56,655,115,378,0.56
8,G,expanded_0_56,543,82,378,0.56
9,E,expanded_0_57,875,105,312,0.57


,stock_code,company_name,fiscal_year,esg_year,rcept_no,section_count,total_word_count,total_char_count,E_seed_count,E_seed_presence_count,...,G_expanded_0_75_count,G_expanded_0_75_presence_count,G_expanded_0_75_count_tfidf_score,G_expanded_0_75_presence_tfidf_score,G_expanded_0_75_share,ESG_expanded_0_75_count,ESG_expanded_0_75_presence_count,ESG_expanded_0_75_count_tfidf_score,ESG_expanded_0_75_presence_tfidf_score,ESG_expanded_0_75_share
0,000020,동화약품,2022,2022,20230315001100,3,7808,37953,0,0,...,102,12,112.960771,18.143439,0.013064,119,16,131.068767,23.132262,0.015241
1,000020,동화약품,2023,2023,20240319000652,3,8065,38467,0,0,...,99,12,107.732147,18.143439,0.012275,115,15,123.965201,21.257321,0.014259
2,000020,동화약품,2024,2024,20250318000739,3,8097,39259,0,0,...,99,12,107.656869,18.143439,0.012227,116,16,125.764865,23.132262,0.014326
3,000040,KR모터스,2022,2022,20230322001182,3,4201,20136,1,1,...,22,6,22.064378,6.064378,0.005237,38,10,39.482257,10.934013,0.009045
4,000040,KR모터스,2023,2023,20240321002062,3,4888,22402,2,2,...,19,6,19.064378,6.064378,0.003887,38,11,40.183446,12.314721,0.007774


In [11]:
def normalize_stock_code(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.extract(r"(\d+)", expand=False)
        .str.zfill(6)
    )


grade_cols = ["stock_code", "fiscal_year", "industry", "esg_year", "esg_grade", "e_grade", "s_grade", "g_grade"]
grade_df = company_master[grade_cols].copy()
grade_df["stock_code"] = normalize_stock_code(grade_df["stock_code"])
all_score_df["stock_code"] = normalize_stock_code(all_score_df["stock_code"])
for col in ["fiscal_year", "esg_year"]:
    grade_df[col] = pd.to_numeric(grade_df[col], errors="coerce").astype("Int64")
    all_score_df[col] = pd.to_numeric(all_score_df[col], errors="coerce").astype("Int64")
all_score_df["esg_year"] = all_score_df["fiscal_year"] + 1
for col in ["esg_grade", "e_grade", "s_grade", "g_grade"]:
    grade_df[f"{col}_num"] = grade_df[col].map(GRADE_MAP)

score_keys = all_score_df[["stock_code", "fiscal_year", "esg_year"]].drop_duplicates()
grade_keys = grade_df[["stock_code", "fiscal_year", "esg_year"]].drop_duplicates()
matched_keys = score_keys.merge(grade_keys, on=["stock_code", "fiscal_year", "esg_year"], how="inner")
print("score key rows:", len(score_keys))
print("grade key rows:", len(grade_keys))
print("matched key rows:", len(matched_keys))
if matched_keys.empty:
    print("Sample score keys:")
    display(score_keys.head(10))
    print("Sample grade keys:")
    display(grade_keys.head(10))

analysis_df = all_score_df.merge(grade_df, on=["stock_code", "fiscal_year", "esg_year"], how="left")
missing_grade_mask = analysis_df["esg_grade_num"].isna()

print("analysis rows:", len(analysis_df))
print("missing esg_grade_num:", missing_grade_mask.sum())
if missing_grade_mask.any():
    print("Sample unmatched merge keys:")
    display(analysis_df.loc[missing_grade_mask, ["stock_code", "fiscal_year", "esg_year"]].drop_duplicates().head(10))
display(analysis_df[["company_name", "stock_code", "fiscal_year", "esg_grade", "esg_grade_num", "total_word_count", "ESG_seed_count", "ESG_seed_count_tfidf_score", "ESG_seed_presence_tfidf_score"]].head())


score key rows: 378
grade key rows: 381
matched key rows: 378
analysis rows: 378
missing esg_grade_num: 0


,company_name,stock_code,fiscal_year,esg_grade,esg_grade_num,total_word_count,ESG_seed_count,ESG_seed_count_tfidf_score,ESG_seed_presence_tfidf_score
0,동화약품,000020,2022,C,1,7808,117,127.288133,19.351628
1,동화약품,000020,2023,C,1,8065,113,120.184567,17.476687
2,동화약품,000020,2024,C,1,8097,114,121.984231,19.351628
3,KR모터스,000040,2022,D,0,4201,38,39.482257,10.934013
4,KR모터스,000040,2023,D,0,4888,38,40.183446,12.314721


In [12]:
from scipy.stats import spearmanr


def spearman_for_feature(data, y_col, x_col):
    tmp = data[[y_col, x_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < 3 or tmp[x_col].nunique() < 2 or tmp[y_col].nunique() < 2:
        return len(tmp), np.nan, np.nan
    rho, p_value = spearmanr(tmp[x_col], tmp[y_col])
    return len(tmp), float(rho), float(p_value)

metric_suffixes = ["count", "presence_count", "count_tfidf_score", "presence_tfidf_score", "share"]
dictionary_labels = ["seed"] + [f"expanded_{threshold_label(theta)}" for theta in THRESHOLDS]

spearman_rows = []
for label in dictionary_labels:
    theta = np.nan if label == "seed" else float(label.split("_")[-2] + "." + label.split("_")[-1])
    for dimension, y_col in [("ESG", "esg_grade_num"), ("E", "e_grade_num"), ("S", "s_grade_num"), ("G", "g_grade_num")]:
        for metric in metric_suffixes:
            x_col = f"{dimension}_{label}_{metric}"
            if x_col not in analysis_df.columns:
                continue
            n, rho, p_value = spearman_for_feature(analysis_df, y_col, x_col)
            spearman_rows.append({
                "dictionary_label": label,
                "threshold": theta,
                "dimension": dimension,
                "grade_col": y_col,
                "metric": metric,
                "feature": x_col,
                "n": n,
                "spearman_rho": rho,
                "p_value": p_value,
            })

spearman_comparison_df = pd.DataFrame(spearman_rows).sort_values(["dimension", "metric", "spearman_rho"], ascending=[True, True, False])
display(spearman_comparison_df)

main_spearman_df = spearman_comparison_df[
    (spearman_comparison_df["dimension"] == "ESG")
    & (spearman_comparison_df["metric"].isin(["count", "presence_count", "count_tfidf_score", "presence_tfidf_score"]))
].sort_values(["metric", "spearman_rho"], ascending=[True, False])

display(main_spearman_df)


,dictionary_label,threshold,dimension,grade_col,metric,feature,n,spearman_rho,p_value
25,expanded_0_55,0.55,E,e_grade_num,count,E_expanded_0_55_count,378,0.530893,7.160677e-29
225,expanded_0_65,0.65,E,e_grade_num,count,E_expanded_0_65_count,378,0.529405,1.084587e-28
165,expanded_0_62,0.62,E,e_grade_num,count,E_expanded_0_62_count,378,0.528653,1.336999e-28
185,expanded_0_63,0.63,E,e_grade_num,count,E_expanded_0_63_count,378,0.528252,1.494458e-28
205,expanded_0_64,0.64,E,e_grade_num,count,E_expanded_0_64_count,378,0.528228,1.504194e-28
...,...,...,...,...,...,...,...,...,...
374,expanded_0_72,0.72,S,s_grade_num,share,S_expanded_0_72_share,378,0.127524,1.309092e-02
14,seed,NaN,S,s_grade_num,share,S_seed_share,378,0.127256,1.328528e-02
394,expanded_0_73,0.73,S,s_grade_num,share,S_expanded_0_73_share,378,0.127256,1.328528e-02
414,expanded_0_74,0.74,S,s_grade_num,share,S_expanded_0_74_share,378,0.127256,1.328528e-02


,dictionary_label,threshold,dimension,grade_col,metric,feature,n,spearman_rho,p_value
0,seed,NaN,ESG,esg_grade_num,count,ESG_seed_count,378,0.719860,1.364310e-61
420,expanded_0_75,0.75,ESG,esg_grade_num,count,ESG_expanded_0_75_count,378,0.718482,2.957679e-61
380,expanded_0_73,0.73,ESG,esg_grade_num,count,ESG_expanded_0_73_count,378,0.718446,3.018421e-61
400,expanded_0_74,0.74,ESG,esg_grade_num,count,ESG_expanded_0_74_count,378,0.718423,3.057701e-61
360,expanded_0_72,0.72,ESG,esg_grade_num,count,ESG_expanded_0_72_count,378,0.718361,3.164550e-61
340,expanded_0_71,0.71,ESG,esg_grade_num,count,ESG_expanded_0_71_count,378,0.716749,7.775289e-61
300,expanded_0_69,0.69,ESG,esg_grade_num,count,ESG_expanded_0_69_count,378,0.715079,1.959782e-60
320,expanded_0_70,0.70,ESG,esg_grade_num,count,ESG_expanded_0_70_count,378,0.714937,2.119527e-60
280,expanded_0_68,0.68,ESG,esg_grade_num,count,ESG_expanded_0_68_count,378,0.714784,2.305966e-60
260,expanded_0_67,0.67,ESG,esg_grade_num,count,ESG_expanded_0_67_count,378,0.714643,2.492049e-60


In [13]:
import statsmodels.api as sm


def zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std


def run_ols(data, y_col, x_cols, min_n=3):
    required_cols = [y_col] + x_cols
    missing_cols = [col for col in required_cols if col not in data.columns]
    if missing_cols:
        return None, {"skip_reason": f"missing columns: {missing_cols}", "n": 0}

    reg_df = data[required_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(reg_df) < min_n:
        return None, {
            "skip_reason": f"too few complete rows after dropna: {len(reg_df)}",
            "n": int(len(reg_df)),
            "missing_by_col": data[required_cols].isna().sum().to_dict(),
        }

    y = reg_df[y_col].astype(float)
    X = reg_df[x_cols].apply(zscore)
    X = sm.add_constant(X, has_constant="add")
    model = sm.OLS(y, X).fit(cov_type="HC3")
    return model, {"skip_reason": "", "n": int(model.nobs)}

ols_rows = []
ols_skip_rows = []
for label in dictionary_labels:
    theta = np.nan if label == "seed" else float(label.split("_")[-2] + "." + label.split("_")[-1])
    for metric in ["count", "presence_count", "count_tfidf_score", "presence_tfidf_score"]:
        x_col = f"ESG_{label}_{metric}"
        if x_col not in analysis_df.columns:
            continue
        model, ols_info = run_ols(analysis_df, "esg_grade_num", [x_col, "total_word_count"])
        if model is None:
            ols_skip_rows.append({
                "dictionary_label": label,
                "threshold": theta,
                "metric": metric,
                "feature": x_col,
                **ols_info,
            })
            continue
        ols_rows.append({
            "dictionary_label": label,
            "threshold": theta,
            "metric": metric,
            "feature": x_col,
            "n": int(model.nobs),
            "coef": float(model.params[x_col]),
            "std_err": float(model.bse[x_col]),
            "p_value": float(model.pvalues[x_col]),
            "r2": float(model.rsquared),
            "word_count_coef": float(model.params["total_word_count"]),
            "word_count_p_value": float(model.pvalues["total_word_count"]),
        })

ols_comparison_df = pd.DataFrame(ols_rows)
if not ols_comparison_df.empty:
    ols_comparison_df = ols_comparison_df.sort_values(["metric", "r2"], ascending=[True, False])
display(ols_comparison_df)

ols_skip_df = pd.DataFrame(ols_skip_rows)
if not ols_skip_df.empty:
    print("Skipped OLS models:", len(ols_skip_df))
    display(ols_skip_df)


,dictionary_label,threshold,metric,feature,n,coef,std_err,p_value,r2,word_count_coef,word_count_p_value
48,expanded_0_66,0.66,count,ESG_expanded_0_66_count,378,1.036877,0.126141,2.035556e-16,0.384350,-0.047744,0.640467
28,expanded_0_61,0.61,count,ESG_expanded_0_61_count,378,1.015220,0.118814,1.289464e-17,0.383816,-0.021564,0.827616
16,expanded_0_58,0.58,count,ESG_expanded_0_58_count,378,1.019206,0.119799,1.774868e-17,0.383185,-0.027564,0.780569
24,expanded_0_60,0.60,count,ESG_expanded_0_60_count,378,1.006010,0.117103,8.634836e-18,0.382686,-0.011815,0.904065
12,expanded_0_57,0.57,count,ESG_expanded_0_57_count,378,1.032981,0.120180,8.304511e-18,0.381884,-0.046749,0.642303
20,expanded_0_59,0.59,count,ESG_expanded_0_59_count,378,1.000753,0.117198,1.353924e-17,0.380898,-0.008109,0.934185
44,expanded_0_65,0.65,count,ESG_expanded_0_65_count,378,1.024020,0.126535,5.833080e-16,0.380763,-0.037394,0.713712
8,expanded_0_56,0.56,count,ESG_expanded_0_56_count,378,1.041537,0.122536,1.898127e-17,0.377200,-0.064481,0.532299
32,expanded_0_62,0.62,count,ESG_expanded_0_62_count,378,1.017088,0.125481,5.251595e-16,0.376900,-0.034891,0.732833
36,expanded_0_63,0.63,count,ESG_expanded_0_63_count,378,1.015098,0.126097,8.270093e-16,0.376607,-0.032892,0.748389


In [14]:
# Summarize threshold-level improvement relative to seed
seed_baseline = spearman_comparison_df[
    (spearman_comparison_df["dictionary_label"] == "seed")
    & (spearman_comparison_df["dimension"] == "ESG")
][["metric", "spearman_rho"]].rename(columns={"spearman_rho": "seed_spearman_rho"})

expanded_main = spearman_comparison_df[
    (spearman_comparison_df["dictionary_label"] != "seed")
    & (spearman_comparison_df["dimension"] == "ESG")
].merge(seed_baseline, on="metric", how="left")
expanded_main["rho_minus_seed"] = expanded_main["spearman_rho"] - expanded_main["seed_spearman_rho"]

analysis_summary_df = expanded_main.sort_values(["metric", "rho_minus_seed"], ascending=[True, False])
display(analysis_summary_df)

valid_spearman_df = analysis_summary_df.dropna(subset=["spearman_rho"])
if valid_spearman_df.empty:
    print("No valid expanded-dictionary Spearman results. Check whether analysis_df has matched ESG grades and non-constant score columns.")
    best_by_metric = pd.DataFrame(columns=["metric", "dictionary_label", "threshold", "feature", "spearman_rho", "seed_spearman_rho", "rho_minus_seed", "p_value"])
else:
    best_idx = valid_spearman_df.groupby("metric")["spearman_rho"].idxmax()
    best_by_metric = valid_spearman_df.loc[best_idx].sort_values("spearman_rho", ascending=False)

print("Best expanded dictionary by ESG Spearman for each metric")
display(best_by_metric[["metric", "dictionary_label", "threshold", "feature", "spearman_rho", "seed_spearman_rho", "rho_minus_seed", "p_value"]])


,dictionary_label,threshold,dimension,grade_col,metric,feature,n,spearman_rho,p_value,seed_spearman_rho,rho_minus_seed
0,expanded_0_75,0.75,ESG,esg_grade_num,count,ESG_expanded_0_75_count,378,0.718482,2.957679e-61,0.719860,-0.001378
1,expanded_0_73,0.73,ESG,esg_grade_num,count,ESG_expanded_0_73_count,378,0.718446,3.018421e-61,0.719860,-0.001414
2,expanded_0_74,0.74,ESG,esg_grade_num,count,ESG_expanded_0_74_count,378,0.718423,3.057701e-61,0.719860,-0.001438
3,expanded_0_72,0.72,ESG,esg_grade_num,count,ESG_expanded_0_72_count,378,0.718361,3.164550e-61,0.719860,-0.001499
4,expanded_0_71,0.71,ESG,esg_grade_num,count,ESG_expanded_0_71_count,378,0.716749,7.775289e-61,0.719860,-0.003111
5,expanded_0_69,0.69,ESG,esg_grade_num,count,ESG_expanded_0_69_count,378,0.715079,1.959782e-60,0.719860,-0.004781
6,expanded_0_70,0.70,ESG,esg_grade_num,count,ESG_expanded_0_70_count,378,0.714937,2.119527e-60,0.719860,-0.004923
7,expanded_0_68,0.68,ESG,esg_grade_num,count,ESG_expanded_0_68_count,378,0.714784,2.305966e-60,0.719860,-0.005076
8,expanded_0_67,0.67,ESG,esg_grade_num,count,ESG_expanded_0_67_count,378,0.714643,2.492049e-60,0.719860,-0.005217
9,expanded_0_55,0.55,ESG,esg_grade_num,count,ESG_expanded_0_55_count,378,0.705889,2.825300e-58,0.719860,-0.013971


Best expanded dictionary by ESG Spearman for each metric


,metric,dictionary_label,threshold,feature,spearman_rho,seed_spearman_rho,rho_minus_seed,p_value
21,count_tfidf_score,expanded_0_73,0.73,ESG_expanded_0_73_count_tfidf_score,0.718696,0.719589,-0.000893,2.623490e-61
0,count,expanded_0_75,0.75,ESG_expanded_0_75_count,0.718482,0.719860,-0.001378,2.957679e-61
42,presence_count,expanded_0_61,0.61,ESG_expanded_0_61_presence_count,0.656518,0.605798,0.050720,5.677912e-48
63,presence_tfidf_score,expanded_0_61,0.61,ESG_expanded_0_61_presence_tfidf_score,0.634548,0.602705,0.031842,5.505818e-44
84,share,expanded_0_63,0.63,ESG_expanded_0_63_share,0.050624,0.032721,0.017903,3.262877e-01


In [15]:
# Year-by-year Spearman robustness check
# Text year and grade year differ by one year: esg_year = fiscal_year + 1.

YEAR_GROUP_COLS = ["fiscal_year", "esg_year"]
YEARLY_METRICS = ["count", "count_tfidf_score", "presence_count", "presence_tfidf_score", "share"]
YEARLY_DIMENSION = "ESG"
YEARLY_GRADE_COL = "esg_grade_num"

if not (analysis_df["esg_year"].astype("Int64") == analysis_df["fiscal_year"].astype("Int64") + 1).all():
    bad_year_pairs = analysis_df.loc[
        analysis_df["esg_year"].astype("Int64") != analysis_df["fiscal_year"].astype("Int64") + 1,
        ["stock_code", "company_name", "fiscal_year", "esg_year"],
    ].head(10)
    raise ValueError("Expected esg_year = fiscal_year + 1. Sample mismatches shown below.")

print("Year alignment check passed: esg_year = fiscal_year + 1")
display(
    analysis_df.groupby(YEAR_GROUP_COLS)
    .size()
    .rename("n")
    .reset_index()
)


def spearman_for_feature(data, y_col, x_col):
    tmp = data[[y_col, x_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < 3 or tmp[x_col].nunique() < 2 or tmp[y_col].nunique() < 2:
        return len(tmp), np.nan, np.nan
    rho, p_value = spearmanr(tmp[x_col], tmp[y_col])
    return len(tmp), float(rho), float(p_value)


yearly_rows = []
for (fiscal_year, esg_year), year_df in analysis_df.groupby(YEAR_GROUP_COLS, dropna=False):
    for label in dictionary_labels:
        theta = np.nan if label == "seed" else float(label.split("_")[-2] + "." + label.split("_")[-1])
        for metric in YEARLY_METRICS:
            feature = f"{YEARLY_DIMENSION}_{label}_{metric}"
            if feature not in year_df.columns:
                continue
            n, rho, p_value = spearman_for_feature(year_df, YEARLY_GRADE_COL, feature)
            yearly_rows.append({
                "fiscal_year": fiscal_year,
                "esg_year": esg_year,
                "dictionary_label": label,
                "threshold": theta,
                "metric": metric,
                "feature": feature,
                "n": n,
                "spearman_rho": rho,
                "p_value": p_value,
            })

yearly_spearman_df = pd.DataFrame(yearly_rows).sort_values(
    ["fiscal_year", "metric", "spearman_rho"],
    ascending=[True, True, False],
).reset_index(drop=True)

display(yearly_spearman_df)

Year alignment check passed: esg_year = fiscal_year + 1


,fiscal_year,esg_year,n
0,2022,2023,126
1,2023,2024,126
2,2024,2025,126


,fiscal_year,esg_year,dictionary_label,threshold,metric,feature,n,spearman_rho,p_value
0,2022,2023,seed,NaN,count,ESG_seed_count,126,0.740906,3.501543e-23
1,2022,2023,expanded_0_73,0.73,count,ESG_expanded_0_73_count,126,0.738563,5.645192e-23
2,2022,2023,expanded_0_74,0.74,count,ESG_expanded_0_74_count,126,0.738563,5.645192e-23
3,2022,2023,expanded_0_75,0.75,count,ESG_expanded_0_75_count,126,0.738562,5.646461e-23
4,2022,2023,expanded_0_72,0.72,count,ESG_expanded_0_72_count,126,0.738408,5.825979e-23
...,...,...,...,...,...,...,...,...,...
325,2024,2025,expanded_0_73,0.73,share,ESG_expanded_0_73_share,126,0.068414,4.465500e-01
326,2024,2025,expanded_0_74,0.74,share,ESG_expanded_0_74_share,126,0.066978,4.561691e-01
327,2024,2025,expanded_0_75,0.75,share,ESG_expanded_0_75_share,126,0.066978,4.561691e-01
328,2024,2025,expanded_0_71,0.71,share,ESG_expanded_0_71_share,126,0.063671,4.787559e-01


In [16]:
# Compare seed vs best expanded dictionary within each year and metric.

seed_yearly = yearly_spearman_df[yearly_spearman_df["dictionary_label"].eq("seed")][
    ["fiscal_year", "esg_year", "metric", "spearman_rho"]
].rename(columns={"spearman_rho": "seed_spearman_rho"})

expanded_yearly = yearly_spearman_df[~yearly_spearman_df["dictionary_label"].eq("seed")].merge(
    seed_yearly,
    on=["fiscal_year", "esg_year", "metric"],
    how="left",
)
expanded_yearly["rho_minus_seed"] = expanded_yearly["spearman_rho"] - expanded_yearly["seed_spearman_rho"]

best_expanded_by_year_metric = (
    expanded_yearly.dropna(subset=["spearman_rho"])
    .sort_values(["fiscal_year", "metric", "spearman_rho"], ascending=[True, True, False])
    .groupby(["fiscal_year", "esg_year", "metric"], as_index=False)
    .head(1)
    .sort_values(["fiscal_year", "metric"])
    .reset_index(drop=True)
)

best_improvement_by_year_metric = (
    expanded_yearly.dropna(subset=["rho_minus_seed"])
    .sort_values(["fiscal_year", "metric", "rho_minus_seed"], ascending=[True, True, False])
    .groupby(["fiscal_year", "esg_year", "metric"], as_index=False)
    .head(1)
    .sort_values(["fiscal_year", "metric"])
    .reset_index(drop=True)
)

print("Best expanded dictionary by absolute Spearman within each fiscal_year/esg_year and metric")
display(best_expanded_by_year_metric[[
    "fiscal_year", "esg_year", "metric", "dictionary_label", "threshold", "feature",
    "n", "spearman_rho", "seed_spearman_rho", "rho_minus_seed", "p_value",
]])

print("Best expanded dictionary by seed improvement within each fiscal_year/esg_year and metric")
display(best_improvement_by_year_metric[[
    "fiscal_year", "esg_year", "metric", "dictionary_label", "threshold", "feature",
    "n", "spearman_rho", "seed_spearman_rho", "rho_minus_seed", "p_value",
]])

Best expanded dictionary by absolute Spearman within each fiscal_year/esg_year and metric


,fiscal_year,esg_year,metric,dictionary_label,threshold,feature,n,spearman_rho,seed_spearman_rho,rho_minus_seed,p_value
0,2022,2023,count,expanded_0_73,0.73,ESG_expanded_0_73_count,126,0.738563,0.740906,-0.002342,5.645192e-23
1,2022,2023,count_tfidf_score,expanded_0_74,0.74,ESG_expanded_0_74_count_tfidf_score,126,0.741928,0.744318,-0.002390,2.838020e-23
2,2022,2023,presence_count,expanded_0_55,0.55,ESG_expanded_0_55_presence_count,126,0.658975,0.609395,0.049580,4.942324e-17
3,2022,2023,presence_tfidf_score,expanded_0_61,0.61,ESG_expanded_0_61_presence_tfidf_score,126,0.644244,0.601511,0.042733,3.998743e-16
4,2022,2023,share,expanded_0_70,0.70,ESG_expanded_0_70_share,126,0.067251,0.061602,0.005649,4.543290e-01
5,2023,2024,count,expanded_0_73,0.73,ESG_expanded_0_73_count,126,0.715040,0.718204,-0.003164,5.228305e-21
6,2023,2024,count_tfidf_score,expanded_0_73,0.73,ESG_expanded_0_73_count_tfidf_score,126,0.722381,0.725801,-0.003421,1.337763e-21
7,2023,2024,presence_count,expanded_0_61,0.61,ESG_expanded_0_61_presence_count,126,0.661451,0.630474,0.030977,3.437674e-17
8,2023,2024,presence_tfidf_score,expanded_0_75,0.75,ESG_expanded_0_75_presence_tfidf_score,126,0.648270,0.630252,0.018018,2.283967e-16
9,2023,2024,share,expanded_0_63,0.63,ESG_expanded_0_63_share,126,0.001997,-0.015370,0.017367,9.822967e-01


Best expanded dictionary by seed improvement within each fiscal_year/esg_year and metric


,fiscal_year,esg_year,metric,dictionary_label,threshold,feature,n,spearman_rho,seed_spearman_rho,rho_minus_seed,p_value
0,2022,2023,count,expanded_0_73,0.73,ESG_expanded_0_73_count,126,0.738563,0.740906,-0.002342,5.645192e-23
1,2022,2023,count_tfidf_score,expanded_0_74,0.74,ESG_expanded_0_74_count_tfidf_score,126,0.741928,0.744318,-0.002390,2.838020e-23
2,2022,2023,presence_count,expanded_0_55,0.55,ESG_expanded_0_55_presence_count,126,0.658975,0.609395,0.049580,4.942324e-17
3,2022,2023,presence_tfidf_score,expanded_0_61,0.61,ESG_expanded_0_61_presence_tfidf_score,126,0.644244,0.601511,0.042733,3.998743e-16
4,2022,2023,share,expanded_0_70,0.70,ESG_expanded_0_70_share,126,0.067251,0.061602,0.005649,4.543290e-01
5,2023,2024,count,expanded_0_73,0.73,ESG_expanded_0_73_count,126,0.715040,0.718204,-0.003164,5.228305e-21
6,2023,2024,count_tfidf_score,expanded_0_73,0.73,ESG_expanded_0_73_count_tfidf_score,126,0.722381,0.725801,-0.003421,1.337763e-21
7,2023,2024,presence_count,expanded_0_61,0.61,ESG_expanded_0_61_presence_count,126,0.661451,0.630474,0.030977,3.437674e-17
8,2023,2024,presence_tfidf_score,expanded_0_75,0.75,ESG_expanded_0_75_presence_tfidf_score,126,0.648270,0.630252,0.018018,2.283967e-16
9,2023,2024,share,expanded_0_63,0.63,ESG_expanded_0_63_share,126,0.001997,-0.015370,0.017367,9.822967e-01


In [17]:
# Compact robustness summary for the main conclusion: seed count vs expanded dictionaries.

seed_count_by_year = yearly_spearman_df[
    yearly_spearman_df["dictionary_label"].eq("seed")
    & yearly_spearman_df["metric"].eq("count")
][["fiscal_year", "esg_year", "n", "spearman_rho", "p_value"]].rename(
    columns={"spearman_rho": "seed_count_rho", "p_value": "seed_count_p_value"}
)

best_expanded_any_metric_by_year = (
    yearly_spearman_df[~yearly_spearman_df["dictionary_label"].eq("seed")]
    .dropna(subset=["spearman_rho"])
    .sort_values(["fiscal_year", "spearman_rho"], ascending=[True, False])
    .groupby(["fiscal_year", "esg_year"], as_index=False)
    .head(1)
    .rename(columns={"spearman_rho": "best_expanded_rho", "p_value": "best_expanded_p_value"})
)

yearly_main_summary = seed_count_by_year.merge(
    best_expanded_any_metric_by_year[[
        "fiscal_year", "esg_year", "dictionary_label", "threshold", "metric", "feature",
        "best_expanded_rho", "best_expanded_p_value",
    ]],
    on=["fiscal_year", "esg_year"],
    how="left",
)
yearly_main_summary["best_expanded_minus_seed_count"] = (
    yearly_main_summary["best_expanded_rho"] - yearly_main_summary["seed_count_rho"]
)

display(yearly_main_summary.sort_values("fiscal_year"))

,fiscal_year,esg_year,n,seed_count_rho,seed_count_p_value,dictionary_label,threshold,metric,feature,best_expanded_rho,best_expanded_p_value,best_expanded_minus_seed_count
0,2022,2023,126,0.740906,3.501543e-23,expanded_0_74,0.74,count_tfidf_score,ESG_expanded_0_74_count_tfidf_score,0.741928,2.838020e-23,0.001022
1,2023,2024,126,0.718204,2.920677e-21,expanded_0_73,0.73,count_tfidf_score,ESG_expanded_0_73_count_tfidf_score,0.722381,1.337763e-21,0.004176
2,2024,2025,126,0.699779,7.790356e-20,expanded_0_75,0.75,count,ESG_expanded_0_75_count,0.701298,5.999576e-20,0.001519
